# Proyecto 1 — Riesgo Crediticio en Fintech
### ETL — Limpieza y normalización de datos

**Grupo:** `Luciana Chutte; Martin Orlando` · **Caso 2:** Organización privada (Finanzas / Banca-Fintech)
**Dataset:** `deudores_tarjeta_credito.csv` — 55.000 clientes de tarjeta de crédito

> Esta notebook toma como insumo la auditoría de
> calidad de datos hecha en `EDA_calidad_datos.ipynb` y aplica las decisiones de
> limpieza que quedaron documentadas ahí como "próximos pasos": normalizar
> `ratio_uso_credito`, e imputar cada variable con faltantes según su clasificación
> MCAR / MAR / MNAR.

## 1. Importación de librerías

In [1]:
import logging
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger("etl")

## 2. Carga de datos

Cargamos el CSV crudo, el mismo insumo que se auditó en el EDA.

In [2]:
df = pd.read_csv('deudores_tarjeta_credito.csv')
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
df.head()

Filas: 55,000
Columnas: 19


,id_cliente,edad,genero,estado_civil,nivel_educativo,situacion_laboral,antiguedad_laboral_meses,ingreso_mensual,limite_credito,saldo_actual,ratio_uso_credito,num_dependientes,num_prestamos_activos,atrasos_ultimos_12m,score_crediticio_previo,deuda_total,ratio_deuda_ingreso,antiguedad_cliente_meses,default_flag
0,100001,46,F,Soltero/a,Universitario,Desempleado,1.0,793857.0,1447430.0,628964.0,435.00,0.0,2,2.0,472.0,733691.0,77.00,27,0
1,100002,38,M,Soltero/a,Universitario,Relacion de dependencia,139.0,625123.0,1712616.0,369782.0,216.00,2.0,2,0.0,706.0,603674.0,0.08,18,0
2,100003,48,F,Soltero/a,Secundario,Jubilado,80.0,235620.0,898166.0,472834.0,526.00,2.0,2,0.0,651.0,798947.0,283.00,71,0
3,100004,58,M,Casado/a,Universitario,Independiente/Monotributista,4.0,1607930.0,4423856.0,796083.0,0.18,3.0,3,0.0,701.0,1833400.0,95.00,42,0
4,100005,37,F,Soltero/a,Posgrado,Relacion de dependencia,22.0,334135.0,1228880.0,822223.0,669.00,NaN,2,0.0,NaN,1404126.0,0.35,29,0


## 3. Validaciones previas a las transformaciones

Antes de tocar nada, confirmamos el estado de calidad relevado en el EDA: unicidad de
`id_cliente`, ausencia de duplicados, sin nulos en columnas de target, y que las
variables categóricas respeten su dominio esperado.

In [3]:
COLUMNAS_TARGET_CRITICAS = ['default_flag', 'deuda_total', 'saldo_actual', 'limite_credito']

CATEGORIAS_ESPERADAS = {
    'genero': {'F', 'M'},
    'estado_civil': {'Soltero/a', 'Casado/a', 'Divorciado/a', 'Viudo/a'},
    'nivel_educativo': {'Secundario', 'Terciario', 'Universitario', 'Posgrado'},
    'situacion_laboral': {
        'Relacion de dependencia',
        'Independiente/Monotributista',
        'Desempleado',
        'Jubilado',
    },
}

def validar(df, etapa):
    print(f"[VALIDATE:{etapa}] id_cliente único: {df['id_cliente'].is_unique}")
    print(f"[VALIDATE:{etapa}] Filas duplicadas: {df.duplicated().sum()}")

    nulos_criticos = df[COLUMNAS_TARGET_CRITICAS].isna().sum().sum()
    print(f"[VALIDATE:{etapa}] Nulos en columnas críticas del target: {nulos_criticos}")

    for col, esperado in CATEGORIAS_ESPERADAS.items():
        encontrado = set(df[col].dropna().unique())
        extra = encontrado - esperado
        if extra:
            print(f"[VALIDATE:{etapa}] {col} tiene categorías inesperadas: {extra}")

    if not df['default_flag'].dropna().isin([0, 1]).all():
        print(f"[VALIDATE:{etapa}] default_flag tiene valores fuera de {{0,1}}.")

    fuera_rango_edad = ((df['edad'] < 18) | (df['edad'] > 100)).sum()
    print(f"[VALIDATE:{etapa}] Filas con edad fuera de rango plausible (18-100): {fuera_rango_edad}")

validar(df, "pre-transform")

[VALIDATE:pre-transform] id_cliente único: True
[VALIDATE:pre-transform] Filas duplicadas: 0
[VALIDATE:pre-transform] Nulos en columnas críticas del target: 0
[VALIDATE:pre-transform] Filas con edad fuera de rango plausible (18-100): 0


## 4. Transformación — Consistencia: normalizar `ratio_uso_credito`

**Hallazgo del EDA:** `ratio_uso_credito` mezcla dos escalas de carga: una parte de las filas está expresada como proporción y otra en una escala aproximadamente mil veces mayor.

Para trabajar con una definición uniforme, recalculamos una nueva columna desde las variables de origen:

`ratio_uso_credito_normalizado = saldo_actual / limite_credito`

La columna original se conserva sin modificar para mantener trazabilidad.


In [4]:
df['ratio_uso_credito_normalizado'] = (df['saldo_actual'] / df['limite_credito']).round(4)

factor_escala = df['ratio_uso_credito'] / df['ratio_uso_credito_normalizado']
n_por_mil = ((factor_escala > 900) & (factor_escala < 1100)).sum()
n_proporcion = ((factor_escala > 0.9) & (factor_escala < 1.1)).sum()
n_otros = len(df) - n_por_mil - n_proporcion

print(f"Filas cargadas 'por mil': {n_por_mil:,}")
print(f"Filas cargadas como proporción directa: {n_proporcion:,}")
print(f"Filas sin encajar en ningún patrón claro: {n_otros}")

df[['ratio_uso_credito', 'ratio_uso_credito_normalizado']].head()

Filas cargadas 'por mil': 49,522
Filas cargadas como proporción directa: 5,477
Filas sin encajar en ningún patrón claro: 1


,ratio_uso_credito,ratio_uso_credito_normalizado
0,435.00,0.4345
1,216.00,0.2159
2,526.00,0.5264
3,0.18,0.1800
4,669.00,0.6691


**Nota — `ratio_deuda_ingreso`:** en el EDA este campo **no** reconstruyó de forma
limpia a partir de `deuda_total / ingreso_mensual` (el factor de escala no se agrupó en 1
ni en 1000, a diferencia de `ratio_uso_credito`). Por eso **no se modifica acá**: corregirlo
sin conocer su fuente/definición real sería inventar un dato.

## 5. Transformación — Faltantes: imputación según clasificación MCAR / MAR / MNAR

Recordando la clasificación:

| Columna | Clasificación | Estrategia | Indicador |
|---|---|---|---|
| `ingreso_mensual` | MAR (según `situacion_laboral`) | Mediana por grupo | `ingreso_mensual_imputado` |
| `score_crediticio_previo` | MAR (según antigüedad como cliente) | Mediana por bucket de antigüedad | `score_crediticio_previo_imputado` |
| `atrasos_ultimos_12m` | MCAR (hipótesis) | Moda | sin indicador |
| `num_dependientes` | MCAR (hipótesis) | Moda | sin indicador |
| `antiguedad_laboral_meses` | MNAR (hipótesis, sin confirmar) | Mediana por grupo, **provisoria** | `antiguedad_laboral_meses_faltante` |

### 5.1 `ingreso_mensual` (MAR) — mediana por `situacion_laboral` + indicador

In [5]:
df['ingreso_mensual_imputado'] = df['ingreso_mensual'].isna().astype(int)
medianas = df.groupby('situacion_laboral')['ingreso_mensual'].transform('median')
df['ingreso_mensual'] = df['ingreso_mensual'].fillna(medianas)
df['ingreso_mensual'] = df['ingreso_mensual'].fillna(df['ingreso_mensual'].median())

print(f"Valores imputados: {df['ingreso_mensual_imputado'].sum():,}")

Valores imputados: 7,132


### 5.2 `score_crediticio_previo` (MAR) — mediana por bucket de antigüedad + indicador

In [6]:
df['score_crediticio_previo_imputado'] = df['score_crediticio_previo'].isna().astype(int)

bucket = pd.cut(
    df['antiguedad_cliente_meses'],
    bins=[0, 12, 24, 48, 100, 1000],
    labels=['0-12m', '12-24m', '24-48m', '48-100m', '100m+'],
)
medianas = df.groupby(bucket, observed=True)['score_crediticio_previo'].transform('median')
df['score_crediticio_previo'] = df['score_crediticio_previo'].fillna(medianas)
df['score_crediticio_previo'] = df['score_crediticio_previo'].fillna(df['score_crediticio_previo'].median())

print(f"Valores imputados: {df['score_crediticio_previo_imputado'].sum():,}")

Valores imputados: 6,939


### 5.3 `atrasos_ultimos_12m` y `num_dependientes` (MCAR, hipótesis) — moda, sin indicador

Al no depender de otra variable observada (según lo relevado en Fase 1), se imputan con la
moda de cada columna. No se agrega columna indicadora, tal como se dejó anotado.

In [7]:
for col in ['atrasos_ultimos_12m', 'num_dependientes']:
    moda = df[col].mode(dropna=True).iloc[0]
    n_faltantes = df[col].isna().sum()
    df[col] = df[col].fillna(moda)
    print(f"{col}: {n_faltantes:,} valores imputados con la moda ({moda}). Sin indicador.")

atrasos_ultimos_12m: 2,675 valores imputados con la moda (0.0). Sin indicador.
num_dependientes: 3,174 valores imputados con la moda (1.0). Sin indicador.


### 5.4 `antiguedad_laboral_meses` (MNAR, hipótesis sin confirmar) — imputación provisoria + indicador

Este es el caso más delicado: la ausencia podría depender del propio valor no observado
(trabajos muy nuevos o precarios que ni se llegan a cargar). No se puede confirmar solo con
los datos, así que **no se asume una imputación definitiva**: se imputa de forma
conservadora (mediana por `situacion_laboral`) para no bloquear el uso de la columna, pero
queda marcada como **pendiente de validar con negocio** vía la columna indicadora.

In [8]:
df['antiguedad_laboral_meses_faltante'] = df['antiguedad_laboral_meses'].isna().astype(int)
medianas = df.groupby('situacion_laboral')['antiguedad_laboral_meses'].transform('median')
df['antiguedad_laboral_meses'] = df['antiguedad_laboral_meses'].fillna(medianas)
df['antiguedad_laboral_meses'] = df['antiguedad_laboral_meses'].fillna(df['antiguedad_laboral_meses'].median())

print(f"Valores imputados (PROVISORIO, pendiente validar con negocio): {df['antiguedad_laboral_meses_faltante'].sum():,}")

Valores imputados (PROVISORIO, pendiente validar con negocio): 4,780


## 6. Transformación — Marcado de inconsistencias semánticas (no se borran)

**Hallazgo del EDA:** clientes con `situacion_laboral = 'Desempleado'` que
igual tienen `antiguedad_laboral_meses` cargada. No es imposible (podría ser antigüedad en
un empleo anterior), así que no se elimina el dato: se agrega un flag para que quede visible
en el modelado y en el dashboard.

Tampoco se recortan los outliers de `ingreso_mensual` y `antiguedad_laboral_meses` — son
variables económicas de cola larga, no errores de carga.

In [9]:
df['flag_desempleado_con_antiguedad_laboral'] = (
    (df['situacion_laboral'] == 'Desempleado') & df['antiguedad_laboral_meses'].notna()
).astype(int)

print(f"Filas marcadas: {df['flag_desempleado_con_antiguedad_laboral'].sum():,}")

Filas marcadas: 5,485


## 7. Validaciones después de transformar

Confirmamos que no queden nulos, que se mantenga la unicidad de `id_cliente` y que las columnas críticas del target sigan intactas.


In [10]:
nulos_restantes = df.isna().sum()
nulos_restantes = nulos_restantes[nulos_restantes > 0]
if len(nulos_restantes):
    print("Columnas con nulos restantes:")
    print(nulos_restantes)
else:
    print("Sin nulos restantes en el dataset.")

validar(df, "post-transform")

Sin nulos restantes en el dataset.
[VALIDATE:post-transform] id_cliente único: True


[VALIDATE:post-transform] Filas duplicadas: 0
[VALIDATE:post-transform] Nulos en columnas críticas del target: 0
[VALIDATE:post-transform] Filas con edad fuera de rango plausible (18-100): 0


## 8. Exportar dataset limpio

Guardamos el dataset transformado como `deudores_tarjeta_credito_clean.csv`, conservando las columnas originales y agregando las variables de normalización, indicadores de imputación y flags de control.


In [11]:
df.to_csv('deudores_tarjeta_credito_clean.csv', index=False)
print(f"Dataset limpio guardado. Dimensiones finales: {df.shape[0]:,} filas x {df.shape[1]} columnas")
df.columns.tolist()

Dataset limpio guardado. Dimensiones finales: 55,000 filas x 24 columnas


['id_cliente',
 'edad',
 'genero',
 'estado_civil',
 'nivel_educativo',
 'situacion_laboral',
 'antiguedad_laboral_meses',
 'ingreso_mensual',
 'limite_credito',
 'saldo_actual',
 'ratio_uso_credito',
 'num_dependientes',
 'num_prestamos_activos',
 'atrasos_ultimos_12m',
 'score_crediticio_previo',
 'deuda_total',
 'ratio_deuda_ingreso',
 'antiguedad_cliente_meses',
 'default_flag',
 'ratio_uso_credito_normalizado',
 'ingreso_mensual_imputado',
 'score_crediticio_previo_imputado',
 'antiguedad_laboral_meses_faltante',
 'flag_desempleado_con_antiguedad_laboral']

## 9. Síntesis del ETL

| Paso | Acción | Resultado |
|---|---|---|
| Consistencia | Recalculado `ratio_uso_credito_normalizado` desde `saldo_actual / limite_credito` | Escala uniforme |
| MAR | `ingreso_mensual` y `score_crediticio_previo` imputados por mediana de grupo | Se conservan indicadores de imputación |
| MCAR | `atrasos_ultimos_12m` y `num_dependientes` imputados con la moda | Sin indicador adicional |
| MNAR | `antiguedad_laboral_meses` imputada de forma provisoria | Indicador de faltante; pendiente validar con negocio |
| Exactitud | Casos de desempleados con antigüedad laboral informada | Se marcan con un flag, sin eliminar filas |
| Sin modificar | `ratio_deuda_ingreso` | Pendiente de validar su definición/fuente con negocio |

**Resultado final:** 55.000 registros, sin valores nulos luego de las transformaciones y manteniendo `id_cliente` como identificador único.

El dataset limpio queda preparado para alimentar el dashboard y, en una etapa posterior, un eventual modelado de riesgo.
